#Proyecto: Control de facturación, cobros y morosidad

Este notebook crea una base de datos SQLite a partir de varios archivos CSV (clientes, productos, facturas, líneas y pagos).
Calcula totales de facturación, cobros, saldos, estado de cada factura y el DSO (Days Sales Outstanding).

In [42]:
# 1) Clonar repo limpio
!rm -rf Portfolio-de-An-lisis-de-Datos
!git clone https://github.com/carmenplata1106/Portfolio-de-An-lisis-de-Datos.git

# 2) Rutas CORRECTAS (nota: proyectos/proyectos)
from pathlib import Path
BASE = Path("/content/Portfolio-de-An-lisis-de-Datos/proyectos/proyectos/06_finanzas_facturacion_cobros")
DATA = BASE / "data"

# 3) Comprobación
import glob, os
print("BASE:", BASE)
print("CSV encontrados:", [os.path.basename(p) for p in glob.glob(str(DATA / "*.csv"))])

# 4) Cargar CSV
import pandas as pd
clientes  = pd.read_csv(DATA / "clientes.csv", parse_dates=["fecha_alta"])
productos = pd.read_csv(DATA / "productos.csv")
facturas  = pd.read_csv(DATA / "facturas.csv", parse_dates=["fecha_emision","fecha_vencimiento"])
lineas    = pd.read_csv(DATA / "lineas_factura.csv")
pagos     = pd.read_csv(DATA / "pagos.csv", parse_dates=["fecha_pago"])
print("✅ Datos cargados")


Cloning into 'Portfolio-de-An-lisis-de-Datos'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (175/175), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 175 (delta 61), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (175/175), 6.70 MiB | 11.34 MiB/s, done.
Resolving deltas: 100% (61/61), done.
BASE: /content/Portfolio-de-An-lisis-de-Datos/proyectos/proyectos/06_finanzas_facturacion_cobros
CSV encontrados: ['clientes.csv', 'pagos.csv', 'productos.csv', 'facturas.csv', 'lineas_factura.csv']
✅ Datos cargados


In [43]:
#Importar librerías necesarias
import pandas as pd
from sqlalchemy import create_engine
from datetime import date
from pathlib import Path

In [44]:
#Definir rutas y conexión a la base de datos
BASE = Path('.')
DATA = BASE / 'data'
engine = create_engine(f'sqlite:///{BASE / 'finanzas.db'}')

print('Conectado a la base de datos:', BASE / 'finanzas.db')

Conectado a la base de datos: finanzas.db


In [45]:

# TRANSFORMACIONES para crear la tabla de facturas completas

# Calcular importe total de cada línea
lineas["importe_linea"] = lineas["cantidad"] * lineas["precio_unitario"]

# Sumar por factura (importe bruto)
totales = (
    lineas.groupby("factura_id", as_index=False)["importe_linea"]
    .sum()
    .rename(columns={"importe_linea": "bruto"})
)

# Unir totales a facturas
fact = facturas.merge(totales, on="factura_id", how="left")
fact["bruto"] = fact["bruto"].fillna(0)

# Calcular neto y total con IVA
fact["neto_sin_iva"] = fact["bruto"] * (1 - fact["descuento_pct"].fillna(0))
fact["total_con_iva"] = fact["neto_sin_iva"] * (1 + fact["iva_pct"].fillna(0))

# Calcular cobros totales por factura
pagos_factura = (
    pagos.groupby("factura_id", as_index=False)["importe_pagado"]
    .sum()
    .rename(columns={"importe_pagado": "cobrado"})
)
fact = fact.merge(pagos_factura, on="factura_id", how="left")
fact["cobrado"] = fact["cobrado"].fillna(0)

# Calcular saldo pendiente
fact["saldo"] = fact["total_con_iva"] - fact["cobrado"]

# Calcular estado de la factura (emitida, parcial, cobrada, vencida)
from datetime import date
import pandas as pd

today = pd.to_datetime(date.today())

def calcular_estado(fila):
    if fila["saldo"] <= 1e-6:
        return "cobrada"
    elif fila["cobrado"] > 0 and fila["saldo"] > 0 and fila["fecha_vencimiento"] >= today:
        return "parcial"
    elif fila["saldo"] > 0 and fila["fecha_vencimiento"] < today:
        return "vencida"
    else:
        return "emitida"

fact["estado_calculado"] = fact.apply(calcular_estado, axis=1)

print("Tabla 'fact' creada con columnas:", fact.columns.tolist())


Tabla 'fact' creada con columnas: ['factura_id', 'cliente_id', 'fecha_emision', 'fecha_vencimiento', 'iva_pct', 'descuento_pct', 'estado', 'bruto', 'neto_sin_iva', 'total_con_iva', 'cobrado', 'saldo', 'estado_calculado']


In [46]:
from sqlalchemy import create_engine
engine = create_engine(f"sqlite:///{(BASE / 'finanzas.db').as_posix()}")

# … (tus transformaciones) …

# Guardar tablas
clientes.to_sql("clientes", engine, if_exists="replace", index=False)
productos.to_sql("productos", engine, if_exists="replace", index=False)
lineas.to_sql("lineas_factura", engine, if_exists="replace", index=False)
pagos.to_sql("pagos", engine, if_exists="replace", index=False)
fact.to_sql("facturas", engine, if_exists="replace", index=False)

print("💾 Creada:", BASE / "finanzas.db")


💾 Creada: finanzas.db


In [47]:
#Calcular totales por factura
lineas['importe_linea'] = lineas['cantidad'] * lineas['precio_unitario']
totales = lineas.groupby('factura_id', as_index=False)['importe_linea'].sum().rename(columns={'importe_linea': 'bruto'})
fact = facturas.merge(totales, on='factura_id', how='left').fillna({'bruto': 0})
fact['neto_sin_iva'] = fact['bruto'] * (1 - fact['descuento_pct'].fillna(0))
fact['total_con_iva'] = fact['neto_sin_iva'] * (1 + fact['iva_pct'].fillna(0))

In [48]:
#Calcular cobros y saldos
pagos_factura = pagos.groupby('factura_id', as_index=False)['importe_pagado'].sum().rename(columns={'importe_pagado': 'cobrado'})
fact = fact.merge(pagos_factura, on='factura_id', how='left').fillna({'cobrado': 0})
fact['saldo'] = fact['total_con_iva'] - fact['cobrado']

In [49]:
#Calcular estado de factura y DSO
today = pd.to_datetime(date.today())

def calcular_estado(f):
    if f['saldo'] <= 1e-6:
        return 'cobrada'
    elif f['cobrado'] > 0 and f['saldo'] > 0 and f['fecha_vencimiento'] >= today:
        return 'parcial'
    elif f['saldo'] > 0 and f['fecha_vencimiento'] < today:
        return 'vencida'
    else:
        return 'emitida'

fact['estado_calculado'] = fact.apply(calcular_estado, axis=1)

pagos_ordenados = pagos.sort_values(['factura_id', 'fecha_pago'])
primer_pago = pagos_ordenados.drop_duplicates('factura_id')[['factura_id', 'fecha_pago']].rename(columns={'fecha_pago': 'fecha_primer_pago'})
fact = fact.merge(primer_pago, on='factura_id', how='left')
fact['dso_dias'] = (fact['fecha_primer_pago'] - fact['fecha_emision']).dt.days

In [50]:
#Guardar las tablas en la base de datos
clientes.to_sql('clientes', engine, if_exists='replace', index=False)
productos.to_sql('productos', engine, if_exists='replace', index=False)
lineas.to_sql('lineas_factura', engine, if_exists='replace', index=False)
pagos.to_sql('pagos', engine, if_exists='replace', index=False)
fact.to_sql('facturas', engine, if_exists='replace', index=False)
print('ETL completado. Se ha creado finanzas.db en la carpeta del proyecto.')

ETL completado. Se ha creado finanzas.db en la carpeta del proyecto.
